In [ ]:
import numpy as np
import pandas as pd
import os
from pathlib import  Path
import h5py


In [ ]:
df_posi=pd.read_csv('df_posi_02—10.csv')


In [ ]:
df_posi

In [ ]:
#订正num
# df_posi.loc[df_posi["lons"] == 111.9217, "num"] = 1
# df_posi[df_posi["lons"]==111.9217]
# df_posi.to_csv('df_posi_02—10.csv',index=False)

In [ ]:
# unique_pairs = df_posi[['indexi', 'indexj']].drop_duplicates()
# num_unique_pairs = len(unique_pairs)

# print(f"Number of unique (indexi, indexj) pairs: {num_unique_pairs}")

In [ ]:
fenbian=0.2
p=int(7/fenbian)
q=int(11/fenbian)
num=np.zeros((p,q))
for i in range(len(df_posi["lons"])):
    x=int((df_posi["lons"][i]-111)/fenbian)
    y=int((df_posi["lats"][i]-30)/fenbian)
    num[y,x]=num[y,x]+1


In [ ]:

num[num>=20].size
# 17568/2

In [ ]:
# merged_df

In [ ]:
# merged_df

In [ ]:
#地基数据
stanum=7255
guage_pre_all=np.full((43848,stanum),np.nan)
nnan=np.full((stanum),np.nan)
dirpath_data = Path(r"H:\202304_202407_guage\per_hour_20_24")
path = r"H:\202304_202407_guage\per_hour_20_24"
files = os.listdir(path)
i = -1  # i为file计数
for file in files:
    print(i,file)
    filepath = dirpath_data / file
    i += 1
    df = pd.read_csv(filepath,header=None, low_memory=False)
    if (int(file[8:18]) >=2024010100) and (int(file[8:18]) <2024071500) and (int(file[8:18]) not in [2024021713,
                                                                                                    2024021918,
                                                                                                    2024022023,
                                                                                                    2024022314,
                                                                                                    2024022322,
                                                                                                    2024022405,
                                                                                                    2024022412,
                                                                                                    2024022513,
                                                                                                    2024022615,
                                                                                                    2024022619,
                                                                                                    2024022818,]):
        df_new=pd.DataFrame()
        df_new["lon"] = df[11]
        df_new["lat"] = df[10]
        df_new["pre"] = df[14]

    else:
        df_new=pd.DataFrame()
        df_new["lon"] = df[3]
        df_new["lat"] = df[2]
        df_new["pre"] = df[6]



    df_new = df_new.drop(index=[0])  # 删第一行
    df_new["lon"] = df_new["lon"].map(float)
    df_new["lat"] = df_new["lat"].map(float)
    df_new["pre"] = df_new["pre"].map(float)
    df_posi_index = df_posi.reset_index().rename(columns={'index': 'posi_index'})#保留原索引

    df_posi_unique=df_posi_index.drop_duplicates(subset=['lons', 'lats'], keep='first')
    df_new_unique=df_new.drop_duplicates(subset=['lon', 'lat'], keep='first')

    merged_df = pd.merge(df_posi_unique, df_new_unique, how='inner', left_on=['lons', 'lats'], right_on=['lon', 'lat'])
    merged_df=merged_df[merged_df['pre'] < 888888].reset_index(drop=True)
    # 保留每组重复项中的第一个
    # unique_df = merged_df.drop_duplicates(subset=['lon', 'lat'], keep='first')

    # 初始化pre数组，赋值为NaN，长度为df_posi中的站点数量
    pre_array = np.full(len(df_posi), np.nan)
    lon_indices = merged_df["posi_index"]

    # 将unique_df['pre']的值按索引放置到pre_array对应位置
    pre_array[lon_indices] = merged_df['pre'].values
    
    # 将pre_array赋值给guage_pre_all的第i行
    guage_pre_all[i, :] = pre_array

    # guage_pre_all[i]= unique_df['pre']

#把得到的降水数据按照索引放在数组中

In [ ]:
# np.save('guage_pre_all.npy',guage_pre_all)

guage_pre_all=np.load('guage_pre_all.npy')

In [ ]:
#加循环储存数据

#卫星数据

#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!这里处理的时候先运行的是2024年11月和12月的数据，在画图的时候要改回来
precip_all=np.zeros((87696,55,35))

dirpath_data = Path(r"H:/2023_6_7IMERGfinal/HOUR/")
path = r"H:/2023_6_7IMERGfinal/HOUR/"
files = os.listdir(path)

i = -1  # i为file计数
for file in files:
    i += 1
    filepath = dirpath_data / file
    print(file)
    with h5py.File(str(filepath), "r") as f:
        precipitation = f["Grid/precipitation"][0,2910:3020,1200:1270]
        result = precipitation.reshape(55, 2, 35, 2).mean(axis=(1, 3))
        precip_all[i]=result
    print(i)

In [ ]:
precip_all_reshaped = precip_all.reshape(-1, 2, precip_all.shape[1], precip_all.shape[2])
precip_all_hour = precip_all_reshaped.mean(axis=1)
precip_all_hour.shape

In [ ]:
np.save('precip_all_hour_satellite.npy',precip_all_hour)
# np.save('guage_pre_all.npy',guage_pre_all)

In [ ]:
guage_pre_all.shape